### setup

In [992]:
from IPython import get_ipython

ip = get_ipython()


def conditional_cells(lines):
    """`# run_if: <expr>` / `# skip_if: <expr>` as the first line gates the cell.

    A comment keeps the cell valid Python, so Pylance/highlighting stay alive
    (any `%%magic` would kill both).
    """
    if lines:
        head = lines[0].strip()
        for tag, want in (("# run_if:", True), ("# skip_if:", False)):
            if head.startswith(tag):
                cond = bool(eval(head[len(tag):], ip.user_ns))
                return lines if cond is want else []
    return lines


# drop any previous copy so re-running this cell doesn't stack transformers
ip.input_transformers_cleanup[:] = [
    t for t in ip.input_transformers_cleanup
    if getattr(t, "__name__", None) != "conditional_cells"
]
ip.input_transformers_cleanup.append(conditional_cells)

In [993]:
import json, re

HEADING = re.compile(r'^(#{1,6})\s+(.+?)\s*#*\s*$')

def headings(path, min_level=1, max_level=6):
    """Yield (cell_index, level, text) for markdown headings in a notebook."""
    nb = json.load(open(path))
    for i, cell in enumerate(nb['cells']):
        if cell['cell_type'] != 'markdown':
            continue
        src = cell['source']
        src = src if isinstance(src, str) else ''.join(src)
        in_fence = False
        for line in src.splitlines():
            if line.lstrip().startswith('```'):
                in_fence = not in_fence
                continue
            if in_fence:
                continue
            m = HEADING.match(line)
            if m and min_level <= len(m.group(1)) <= max_level:
                yield i, len(m.group(1)), m.group(2)

def heading_tree(path, min_level=1, max_level=6):
    """Nest the headings into {text: {child_text: {...}}}."""
    tree, stack = {}, []          # stack of (level, node)
    for _, level, text in headings(path, min_level, max_level):
        while stack and stack[-1][0] >= level:
            stack.pop()
        parent = stack[-1][1] if stack else tree
        node = {}
        parent[text] = node
        stack.append((level, node))
    return tree

In [994]:
from pathlib import Path

NB_PATH = Path(globals()['__vsc_ipynb_file__'])
NB_PATH

PosixPath('/home/rahul/Documents/codes/ml/pytorch-practice/nb/rl/final/value_and_policy_iteration.ipynb')

In [995]:
list(headings(NB_PATH))

[(0, 3, 'setup'),
 (8, 3, 'start'),
 (11, 3, 'Environment'),
 (18, 3, 'value iteration'),
 (28, 3, 'read policy during value_iteration'),
 (33, 3, 'policy iteration'),
 (40, 3, 'analytic_policy_value'),
 (44, 3, 'step reward -1, without discount produces steps to goal behaviour'),
 (50,
  3,
  'step reward -1, value_iteration V settles at -1/(1-gamma) for cells trapped in walls'),
 (55,
  3,
  'step reward -1, policy_evaluation hits against the ceiling similar effect as being trapped in walls'),
 (60,
  3,
  'if both negative step reward and discount is removed, then value_iteration policy degenerates'),
 (66, 3, "don't update action until better is available"),
 (72,
  3,
  'policy iteration keeps flipping up/right on cell (0, 2) without pass_incumbent_policy'),
 (76, 3, 'pass previous V to policy_evaluation')]

In [996]:
heading_tree(NB_PATH)

{'setup': {},
 'start': {},
 'Environment': {},
 'value iteration': {},
 'read policy during value_iteration': {},
 'policy iteration': {},
 'analytic_policy_value': {},
 'step reward -1, without discount produces steps to goal behaviour': {},
 'step reward -1, value_iteration V settles at -1/(1-gamma) for cells trapped in walls': {},
 'step reward -1, policy_evaluation hits against the ceiling similar effect as being trapped in walls': {},
 'if both negative step reward and discount is removed, then value_iteration policy degenerates': {},
 "don't update action until better is available": {},
 'policy iteration keeps flipping up/right on cell (0, 2) without pass_incumbent_policy': {},
 'pass previous V to policy_evaluation': {}}

In [997]:
all_headings = [v for _, _, v in headings(NB_PATH)]
all_headings

['setup',
 'start',
 'Environment',
 'value iteration',
 'read policy during value_iteration',
 'policy iteration',
 'analytic_policy_value',
 'step reward -1, without discount produces steps to goal behaviour',
 'step reward -1, value_iteration V settles at -1/(1-gamma) for cells trapped in walls',
 'step reward -1, policy_evaluation hits against the ceiling similar effect as being trapped in walls',
 'if both negative step reward and discount is removed, then value_iteration policy degenerates',
 "don't update action until better is available",
 'policy iteration keeps flipping up/right on cell (0, 2) without pass_incumbent_policy',
 'pass previous V to policy_evaluation']

In [998]:
import ipywidgets as widgets
from IPython.display import HTML, display

# selection lives on disk, so `Run All` (which rebuilds the picker) doesn't reset it
SELECTION_PATH = NB_PATH.with_suffix('.sections.json')
try:
    SELECTED = set(json.loads(SELECTION_PATH.read_text()))
except (OSError, ValueError):
    SELECTED = set()

PICKER_CSS = """
<style>
/* VS Code paints a white panel behind every widget output; its padding was the white
   frame around the picker, and it forced the native checkboxes to render light. */
.cell-output-ipywidget-background { background-color: transparent !important; }
.jp-OutputArea-output { background-color: transparent !important; }

.section-picker {
    color-scheme: dark;          /* dark native checkbox squares instead of white ones */
    background: #1e1e1e;
    border: 1px solid #3a3a3a;
    border-radius: 6px;
    padding: 10px 14px;
    color: #d4d4d4;
}
.section-picker label,
.section-picker .widget-label,
.section-picker .widget-checkbox > * {
    color: #d4d4d4 !important;
    font-size: 13px;
}
.section-picker input[type="checkbox"] {
    accent-color: #4ea1ff;       /* recolors the tick, no custom sprite needed */
    width: 14px;
    height: 14px;
    margin-right: 6px;
}
.section-picker .jupyter-button:hover {
    background: #3a3a3a !important;
}
</style>
"""

_boxes = {}


def _on_toggle(change):
    text = change['owner'].description
    SELECTED.add(text) if change['new'] else SELECTED.discard(text)
    SELECTION_PATH.write_text(json.dumps(sorted(SELECTED), indent=2))


def _rows(tree, level):
    out = []
    for text, children in tree.items():
        box = widgets.Checkbox(
            value=text in SELECTED, description=text, indent=False,
            layout=widgets.Layout(width='auto', margin=f'2px 0 2px {level * 24}px'),
        )
        box.observe(_on_toggle, names='value')
        _boxes[text] = box
        out.append(box)
        if children:
            out.append(_rows(children, level + 1))
    return widgets.VBox(out)


def _dark_button(description, on_click):
    b = widgets.Button(description=description, layout=widgets.Layout(width='70px'))
    b.style.button_color = '#2d2d2d'   # native traits, no CSS needed for these
    b.style.text_color = '#d4d4d4'
    b.on_click(on_click)
    return b


def section_picker():
    """Checkbox tree over heading_tree(NB_PATH); ticked sections are the ones that run."""
    # its own output, so the rules reach VS Code's wrapper element, which sits
    # *outside* the widget subtree
    display(HTML(PICKER_CSS))

    _boxes.clear()
    body = _rows(heading_tree(NB_PATH), 0)

    def _set_all(v):
        return lambda _: [setattr(b, 'value', v) for b in _boxes.values()]

    controls = widgets.HBox([
        _dark_button('all', _set_all(True)),
        _dark_button('none', _set_all(False)),
    ], layout=widgets.Layout(margin='0 0 8px 0'))

    picker = widgets.VBox([controls, body])
    picker.add_class('section-picker')
    return picker


def cell_allowed():
    if h3 not in all_headings:
        raise ValueError(f'h3={h3!r} matches no heading; expected one of {all_headings}')
    return h3 in SELECTED

### start

In [999]:
section_picker()

In [1000]:
import numpy as np

### Environment

In [1001]:
h3 = 'Environment'

In [1002]:
UP, RIGHT, DOWN, LEFT = 0, 1, 2, 3
DIRS = {UP: (-1, 0), RIGHT: (0, 1), DOWN: (1, 0), LEFT: (0, -1)}
ARROWS = {UP: "↑", RIGHT: "→", DOWN: "↓", LEFT: "←"}

In [1003]:
class GridWorld:

    def __init__(
        self,
        rows: int,
        cols: int,
        step_reward: float,
        terminals: dict[tuple[int, int], float],
        walls: set[tuple[int, int]],
    ):
        self.rows = rows
        self.cols = cols
        self.step_reward = step_reward
        self.terminals = terminals
        self.walls = walls
        self.s2c = [
            (r, c)
            for r in range(self.rows)
            for c in range(self.cols)
            if (r, c) not in self.walls
        ]
        self.nS = len(self.s2c)
        self.nA = 4
        self.c2s = {cell: i for i, cell in enumerate(self.s2c)}

    def step(self, s: int, a: int):
        dr, dc = DIRS[a]
        cell = self.s2c[s]
        nxt = (cell[0] + dr, cell[1] + dc)
        if (
            not 0 <= nxt[0] < self.rows
            or not 0 <= nxt[1] < self.cols
            or nxt in self.walls
        ):
            nxt = cell
        reward = self.terminals.get(nxt, self.step_reward)
        return self.c2s[nxt], reward
    
    def cell_repr(self, r, c):
        cell = (r, c)
        if cell in self.terminals:
            return f'{self.terminals[cell]:+}'
        elif cell in self.walls:
            return '#'
        else:
            return '·'
        
    def render(self):
        for r in range(self.rows):
            for c in range(self.cols):
                if r == 0 and c == 0:
                    print(" r/c", end="")
                    print(''.join([f'{v:>3} ' for v in range(self.cols)]))
                if c == 0:
                    print(f'{r:>3} ', end="")
                print(f'{self.cell_repr(r, c):>3} ', end="")
            print()
            
    def __repr__(self):
        return f'''
Grid(
    rows={self.rows},
    cols={self.cols},
    step_reward={self.step_reward},
    terminals={self.terminals},
    walls={self.walls},
)
'''

In [1004]:
env = GridWorld(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
)

In [1005]:
# run_if: cell_allowed()
env


Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
)

In [1006]:
# run_if: cell_allowed()
env.render()

 r/c  0   1   2   3 
  0   ·   ·   ·  +1 
  1   ·   #   ·  -1 
  2   ·   ·   ·   · 


### value iteration

In [1007]:
h3 = 'value iteration'

In [1008]:
def show_V(env: GridWorld, V: np.ndarray):
    for r in range(env.rows):
        for c in range(env.cols):
            if r == 0 and c == 0:
                print("  r/c", end="")
                print(''.join([f'{v:>4} ' for v in range(env.cols)]))
            if c == 0:
                print(f'{r:>4} ', end="")            
            cell = (r, c)
            if cell in env.c2s:
                value = round(V[env.c2s[(r, c)]], 2)
            else:
                value = '#'
            print(f'{value:>4} ', end="")
        print()

def q_from_v(
    env: GridWorld,
    V: np.ndarray,
    s: int,
    gamma: float,
):
    q = np.zeros(env.nA)
    if env.s2c[s] in env.terminals:
        return q
    for a in range(env.nA):
        ns, r = env.step(s, a)
        q[a] = r + gamma * V[ns]
    return q

def value_iteration(
    env: GridWorld,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    verbose=0,
):
    V = np.zeros(env.nS)
    if verbose >= 2:
        print('---------- value_iteration ------------')
        print('V init')
        show_V(env, V)
        print('-' * 25)    
    delta = float('inf')
    i = 0
    while delta >= theta and i < max_iters:
        delta = 0.0
        V_old = V.copy()
        for s in range(env.nS):
            if env.s2c[s] in env.terminals:
                continue
            V[s] = np.max(q_from_v(env, V_old, s, gamma))
            delta = max(delta, abs(V[s] - V_old[s]))
        if verbose >= 1:
            print(f"iter {i}: delta={delta:.6f}")
        if verbose >= 2:
            show_V(env, V)
            print('-' * 25)
        i += 1
    converged = delta < theta
    if verbose >= 1:
        if converged:
                print(f'value_iteration converged in {i - 1} iterations')
        else:
            print(f"value_iteration did not converge in {max_iters} iterations")
    return V, converged

In [1009]:
# run_if: cell_allowed()
V, converged = value_iteration(env)
V

array([0.81  , 0.9   , 1.    , 0.    , 0.729 , 0.9   , 0.    , 0.6561,
       0.729 , 0.81  , 0.729 ])

In [1010]:
# run_if: cell_allowed()
show_V(env, V)

  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 
   2 0.66 0.73 0.81 0.73 


In [1011]:
# run_if: cell_allowed()
value_iteration(env, verbose=2);

---------- value_iteration ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 1: delta=0.900000
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0    #  0.9  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 2: delta=0.810000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1  0.0    #  0.9  0.0 
   2  0.0  0.0 0.81  0.0 
-------------------------
iter 3: delta=0.729000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 
   2  0.0 0.73 0.81 0.73 
-------------------------
iter 4: delta=0.656100
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 
   2 0.66 0.73 0.81 0.73 
-------------------------
iter 5: delta=0.000000
  r/c   0    1    2    3 
   0 0.81

In [1012]:
def read_policy(
    env: GridWorld,
    V: np.ndarray,
    gamma=0.9,
):
    policy = np.full(env.nS, -1)
    for s in range(env.nS):
        if env.s2c[s] in env.terminals:
            continue
        best_a = int(np.argmax(q_from_v(env, V, s, gamma)))
        policy[s] = best_a
    return policy

In [1013]:
# run_if: cell_allowed()
policy = read_policy(env, V)
policy

array([ 1,  1,  1, -1,  0,  0, -1,  0,  1,  0,  3])

In [1014]:
def render_policy(env: GridWorld, policy: np.ndarray):
    for r in range(env.rows):
        for c in range(env.cols):
            if r == 0 and c == 0:
                print(" r/c", end="")
                print(''.join([f'{v:>3} ' for v in range(env.cols)]))
            if c == 0:
                print(f'{r:>3} ', end="")  

            cell = (r, c)
            if cell in env.c2s:
                if cell in env.terminals:
                    value = f'{env.terminals[cell]:+}'
                else:
                    value = ARROWS[policy[env.c2s[(r, c)]]]
            else:
                value = '#'
            print(f'{value:>3} ', end='')
        print()    

In [1015]:
# run_if: cell_allowed()
render_policy(env, policy)

 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 


### read policy during value_iteration 

we can also read policy during value_iteration, no need to do it with returned V

In [1016]:
h3 = 'read policy during value_iteration'

In [1017]:
def value_iteration(
    env: GridWorld,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    verbose=0,
):
    V = np.zeros(env.nS)
    policy = np.full(env.nS, -1)
    if verbose >= 2:
        print('---------- value_iteration ------------')
        print('V init')
        show_V(env, V)
        print('-' * 25)    
    delta = float('inf')
    i = 0
    while delta >= theta and i < max_iters:
        delta = 0.0
        V_old = V.copy()
        for s in range(env.nS):
            if env.s2c[s] in env.terminals:
                continue
            q = q_from_v(env, V_old, s, gamma)
            V[s] = np.max(q)
            policy[s] = int(np.argmax(q))
            delta = max(delta, abs(V[s] - V_old[s]))
        if verbose >= 1:
            print(f"iter {i}: delta={delta:.6f}")
        if verbose >= 2:
            show_V(env, V)
            print('-' * 25)
        i += 1
    converged = delta < theta
    if verbose >= 1:
        if converged:
                print(f'value_iteration converged in {i - 1} iterations')
        else:
            print(f"value_iteration did not converge in {max_iters} iterations")
    return policy, V, converged

In [1018]:
# run_if: cell_allowed()
policy, V, converged = value_iteration(env)
policy

array([ 1,  1,  1, -1,  0,  0, -1,  0,  1,  0,  3])

In [1019]:
# run_if: cell_allowed()
render_policy(env, policy)

 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 


### policy iteration

In [1020]:
h3 = 'policy iteration'

In [1021]:
def policy_evaluation(
    env: GridWorld,
    policy: np.ndarray,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    verbose=0,
):
    V = np.zeros(env.nS)
    if verbose >= 2:
        print('---------- policy_evaluation ------------')
        print('V init')
        show_V(env, V)
        print('-' * 25)    
    delta = float('inf')
    i = 0
    while delta >= theta and i < max_iters:
        delta = 0.0
        V_old = V.copy()
        for s in range(env.nS):
            if env.s2c[s] in env.terminals:
                continue
            a = policy[s]
            ns, r = env.step(s, a)
            V[s] = r + gamma * V_old[ns]
            delta = max(delta, abs(V[s] - V_old[s]))
        if verbose >= 1:
            print(f"iter {i}: delta={delta:.6f}")
        if verbose >= 2:
            show_V(env, V)
            print('-' * 25)
        i += 1
    converged = delta < theta
    if verbose >= 1:
        if converged:
            print(f'policy_evaluation converged in {i - 1} iterations')
        else:
            print(f"policy_evaluation did not converge in {max_iters} iterations")
    return V, converged

In [1022]:
# run_if: cell_allowed()
policy, *_ = value_iteration(env)
V, converged = policy_evaluation(env, policy, verbose=2)

---------- policy_evaluation ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 1: delta=0.900000
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0    #  0.9  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 2: delta=0.810000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1  0.0    #  0.9  0.0 
   2  0.0  0.0 0.81  0.0 
-------------------------
iter 3: delta=0.729000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 
   2  0.0 0.73 0.81 0.73 
-------------------------
iter 4: delta=0.656100
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 
   2 0.66 0.73 0.81 0.73 
-------------------------
iter 5: delta=0.000000
  r/c   0    1    2    3 
   0 0.

In [1023]:
def policy_iteration(
    env: GridWorld,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    verbose=0,
    pe_max_iters=1000,
    pe_verbose=0,
):
    policy = np.full(env.nS, UP)
    for cell in env.terminals:
        policy[env.c2s[cell]] = -1
    V = np.zeros(env.nS)

    if verbose >= 2:
        print('---------- policy_iteration ------------')
        print('policy init')
        render_policy(env, policy)
    
    i = 0
    pe_converged = True
    changed = None
    while changed != 0 and i < max_iters:
        V, pe_converged = policy_evaluation(
            env, policy, gamma, theta, max_iters=pe_max_iters, verbose=pe_verbose
        )
        if not pe_converged:
            break
        new_policy = read_policy(env, V, gamma)
        changed = int(np.count_nonzero(new_policy != policy))
        if verbose >= 2:
            print('-' * 25)
            print(f'iter {i}: V')
            show_V(env, V)
            print('-' * 5)  
        if verbose >= 1:
            prefix = f'iter {i}: ' if verbose == 1 else ''
            print(f'{prefix}actions changed = {changed}')
        if verbose >= 2:
            render_policy(env, new_policy)
        policy = new_policy
        i += 1
    converged = changed == 0
    if verbose >= 1:
        print('-' * 25)  
        if not pe_converged:
            print(
                f"policy_iteration did not converge because policy_evaluation did not converge"
            )
        elif converged:
            print(f'policy_iteration converged in {i - 1} iterations')
        else:
            print(f'policy_iteration did not converge in {max_iters} iterations')
    return policy, V, converged    

In [1024]:
# run_if: cell_allowed()
policy, V, converged = policy_iteration(env, verbose=2)

---------- policy_iteration ------------
policy init
 r/c  0   1   2   3 
  0   ↑   ↑   ↑  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ↑ 
-------------------------
iter 0: V
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-----
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ← 
-------------------------
iter 1: V
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0    #  0.9  0.0 
   2  0.0  0.0 0.81 0.73 
-----
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 
-------------------------
iter 2: V
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0    #  0.9  0.0 
   2  0.0 0.73 0.81 0.73 
-----
actions changed = 2
 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   →   →   ↑   ← 
-------------------------
iter 3: V
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 

In [1025]:
# run_if: cell_allowed()
policy, V, converged = policy_iteration(env, verbose=2, pe_verbose=2)

---------- policy_iteration ------------
policy init
 r/c  0   1   2   3 
  0   ↑   ↑   ↑  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ↑ 
---------- policy_evaluation ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
iter 1: delta=0.000000
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
policy_evaluation converged in 1 iterations
-------------------------
iter 0: V
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-----
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ← 
---------- policy_evaluation ------------
V init
  r/c   0    1    2    3 
   0  0.0

### analytic_policy_value

In [1026]:
h3 = "analytic_policy_value"

In [1027]:
def analytic_policy_value(env: GridWorld, policy: np.ndarray, gamma=0.9):
    n = env.nS
    Ppi = np.zeros((n, n))
    rpi = np.zeros(n)
    for s in range(n):
        a = policy[s]
        if a == -1:
            continue
        pa = 1.0
        ns, r = env.step(s, a)
        prob = 1.0
        Ppi[s, ns] += pa * prob
        rpi[s] += pa * prob * r
    return np.linalg.solve(np.eye(n) - gamma * Ppi, rpi)

In [1028]:
# run_if: cell_allowed()
policy, V_star, converged = value_iteration(env)
V = analytic_policy_value(env, policy, 0.9)
show_V(env, V)
print('-' * 25)
equal = np.array_equal(V, V_star)
print(f'V derived by analytic_policy_value equal to V_star from value_iteration = {equal}')

  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 
   2 0.66 0.73 0.81 0.73 
-------------------------
V derived by analytic_policy_value equal to V_star from value_iteration = True


### step reward -1, without discount produces steps to goal behaviour 

In [1029]:
h3 = 'step reward -1, without discount produces steps to goal behaviour'

In [1030]:
# run_if: cell_allowed()
env = GridWorld(
    rows=3,
    cols=4,
    step_reward=-1,
    terminals={(0, 3): -1},
    walls={(1, 1)},
)

In [1031]:
# run_if: cell_allowed()
policy, V, converged = value_iteration(env, gamma=1.0, verbose=2)
print('-' * 25)
render_policy(env, policy)

---------- value_iteration ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0 -1.0 -1.0 -1.0  0.0 
   1 -1.0    # -1.0 -1.0 
   2 -1.0 -1.0 -1.0 -1.0 
-------------------------
iter 1: delta=1.000000
  r/c   0    1    2    3 
   0 -2.0 -2.0 -1.0  0.0 
   1 -2.0    # -2.0 -1.0 
   2 -2.0 -2.0 -2.0 -2.0 
-------------------------
iter 2: delta=1.000000
  r/c   0    1    2    3 
   0 -3.0 -2.0 -1.0  0.0 
   1 -3.0    # -2.0 -1.0 
   2 -3.0 -3.0 -3.0 -2.0 
-------------------------
iter 3: delta=1.000000
  r/c   0    1    2    3 
   0 -3.0 -2.0 -1.0  0.0 
   1 -4.0    # -2.0 -1.0 
   2 -4.0 -4.0 -3.0 -2.0 
-------------------------
iter 4: delta=1.000000
  r/c   0    1    2    3 
   0 -3.0 -2.0 -1.0  0.0 
   1 -4.0    # -2.0 -1.0 
   2 -5.0 -4.0 -3.0 -2.0 
-------------------------
iter 5: delta=0.000000
  r/c   0    1    2    3 
   0 -3.0

policy_evaluation will fail to converge for all UP policy in above case

In [1032]:
# run_if: cell_allowed()
policy, V, converged = policy_iteration(env, gamma=1.0, verbose=2, pe_verbose=2)

---------- policy_iteration ------------
policy init
 r/c  0   1   2   3 
  0   ↑   ↑   ↑  -1 
  1   ↑   #   ↑   ↑ 
  2   ↑   ↑   ↑   ↑ 
---------- policy_evaluation ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0 -1.0 -1.0 -1.0  0.0 
   1 -1.0    # -1.0 -1.0 
   2 -1.0 -1.0 -1.0 -1.0 
-------------------------
iter 1: delta=1.000000
  r/c   0    1    2    3 
   0 -2.0 -2.0 -2.0  0.0 
   1 -2.0    # -2.0 -1.0 
   2 -2.0 -2.0 -2.0 -2.0 
-------------------------
iter 2: delta=1.000000
  r/c   0    1    2    3 
   0 -3.0 -3.0 -3.0  0.0 
   1 -3.0    # -3.0 -1.0 
   2 -3.0 -3.0 -3.0 -2.0 
-------------------------
iter 3: delta=1.000000
  r/c   0    1    2    3 
   0 -4.0 -4.0 -4.0  0.0 
   1 -4.0    # -4.0 -1.0 
   2 -4.0 -4.0 -4.0 -2.0 
-------------------------
iter 4: delta=1.000000
  r/c   0    1    2    3 
   0 -5.0 -5.0 -5.0  0.

### step reward -1, value_iteration V settles at -1/(1-gamma) for cells trapped in walls

In [1033]:
h3 = 'step reward -1, value_iteration V settles at -1/(1-gamma) for cells trapped in walls'

In [1034]:
# run_if: cell_allowed()
env = GridWorld(
    rows=5,
    cols=6,
    step_reward=-1,
    terminals={(0, 5): -1},
    walls={(2, 2), (2, 3), (3, 1), (3, 4), (4, 2), (4, 3)},
)
env.render()

 r/c  0   1   2   3   4   5 
  0   ·   ·   ·   ·   ·  -1 
  1   ·   ·   ·   ·   ·   · 
  2   ·   ·   #   #   ·   · 
  3   ·   #   ·   ·   #   · 
  4   ·   ·   #   #   ·   · 


In [1035]:
# run_if: cell_allowed()
policy, V, converged = value_iteration(env, verbose=2)

---------- value_iteration ------------
V init
  r/c   0    1    2    3    4    5 
   0  0.0  0.0  0.0  0.0  0.0  0.0 
   1  0.0  0.0  0.0  0.0  0.0  0.0 
   2  0.0  0.0    #    #  0.0  0.0 
   3  0.0    #  0.0  0.0    #  0.0 
   4  0.0  0.0    #    #  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3    4    5 
   0 -1.0 -1.0 -1.0 -1.0 -1.0  0.0 
   1 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 
   2 -1.0 -1.0    #    # -1.0 -1.0 
   3 -1.0    # -1.0 -1.0    # -1.0 
   4 -1.0 -1.0    #    # -1.0 -1.0 
-------------------------
iter 1: delta=0.900000
  r/c   0    1    2    3    4    5 
   0 -1.9 -1.9 -1.9 -1.9 -1.0  0.0 
   1 -1.9 -1.9 -1.9 -1.9 -1.9 -1.0 
   2 -1.9 -1.9    #    # -1.9 -1.9 
   3 -1.9    # -1.9 -1.9    # -1.9 
   4 -1.9 -1.9    #    # -1.9 -1.9 
-------------------------
iter 2: delta=0.810000
  r/c   0    1    2    3    4    5 
   0 -2.71 -2.71 -2.71 -1.9 -1.0  0.0 
   1 -2.71 -2.71 -2.71 -2.71 -1.9 -1.0 
   2 -2.71 -2.71    #    # -2.71 -1.9 
   3

In [1036]:
# run_if: cell_allowed()
render_policy(env, policy)

 r/c  0   1   2   3   4   5 
  0   →   →   →   →   →  -1 
  1   ↑   ↑   ↑   ↑   ↑   ↑ 
  2   ↑   ↑   #   #   ↑   ↑ 
  3   ↑   #   ↑   ↑   #   ↑ 
  4   ↑   ←   #   #   →   ↑ 


### step reward -1, policy_evaluation hits against the ceiling similar effect as being trapped in walls

In [1037]:
h3 = 'step reward -1, policy_evaluation hits against the ceiling similar effect as being trapped in walls'

In [1038]:
# run_if: cell_allowed()
env = GridWorld(
    rows=3,
    cols=4,
    step_reward=-1,
    terminals={(0, 3): -1},
    walls={(1, 1)},
)
env.render()

 r/c  0   1   2   3 
  0   ·   ·   ·  -1 
  1   ·   #   ·   · 
  2   ·   ·   ·   · 


In [1039]:
# run_if: cell_allowed()
policy, V, converged = policy_iteration(env, verbose=2)

---------- policy_iteration ------------
policy init
 r/c  0   1   2   3 
  0   ↑   ↑   ↑  -1 
  1   ↑   #   ↑   ↑ 
  2   ↑   ↑   ↑   ↑ 
-------------------------
iter 0: V
  r/c   0    1    2    3 
   0 -10.0 -10.0 -10.0  0.0 
   1 -10.0    # -10.0 -1.0 
   2 -10.0 -10.0 -10.0 -1.9 
-----
actions changed = 3
 r/c  0   1   2   3 
  0   ↑   ↑   →  -1 
  1   ↑   #   →   ↑ 
  2   ↑   ↑   →   ↑ 
-------------------------
iter 1: V
  r/c   0    1    2    3 
   0 -10.0 -10.0 -1.0  0.0 
   1 -10.0    # -1.9 -1.0 
   2 -10.0 -10.0 -2.71 -1.9 
-----
actions changed = 4
 r/c  0   1   2   3 
  0   ↑   →   →  -1 
  1   ↑   #   ↑   ↑ 
  2   ↑   →   ↑   ↑ 
-------------------------
iter 2: V
  r/c   0    1    2    3 
   0 -10.0 -1.9 -1.0  0.0 
   1 -10.0    # -1.9 -1.0 
   2 -10.0 -3.44 -2.71 -1.9 
-----
actions changed = 2
 r/c  0   1   2   3 
  0   →   →   →  -1 
  1   ↑   #   ↑   ↑ 
  2   →   →   ↑   ↑ 
-------------------------
iter 3: V
  r/c   0    1    2    3 
   0 -2.71 -1.9 -1.0  0.0 
   1 

In [1040]:
# run_if: cell_allowed()
policy, V, converged = policy_iteration(env, verbose=2, pe_verbose=2)

---------- policy_iteration ------------
policy init
 r/c  0   1   2   3 
  0   ↑   ↑   ↑  -1 
  1   ↑   #   ↑   ↑ 
  2   ↑   ↑   ↑   ↑ 
---------- policy_evaluation ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0 -1.0 -1.0 -1.0  0.0 
   1 -1.0    # -1.0 -1.0 
   2 -1.0 -1.0 -1.0 -1.0 
-------------------------
iter 1: delta=0.900000
  r/c   0    1    2    3 
   0 -1.9 -1.9 -1.9  0.0 
   1 -1.9    # -1.9 -1.0 
   2 -1.9 -1.9 -1.9 -1.9 
-------------------------
iter 2: delta=0.810000
  r/c   0    1    2    3 
   0 -2.71 -2.71 -2.71  0.0 
   1 -2.71    # -2.71 -1.0 
   2 -2.71 -2.71 -2.71 -1.9 
-------------------------
iter 3: delta=0.729000
  r/c   0    1    2    3 
   0 -3.44 -3.44 -3.44  0.0 
   1 -3.44    # -3.44 -1.0 
   2 -3.44 -3.44 -3.44 -1.9 
-------------------------
iter 4: delta=0.656100
  r/c   0    1    2    3 
   0 -4

### if both negative step reward and discount is removed, then value_iteration policy degenerates

In [1041]:
h3 = 'if both negative step reward and discount is removed, then value_iteration policy degenerates'

In [1042]:
# run_if: cell_allowed()
env = GridWorld(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1},
    walls={(1, 1)},
)
env.render()

 r/c  0   1   2   3 
  0   ·   ·   ·  +1 
  1   ·   #   ·   · 
  2   ·   ·   ·   · 


In [1043]:
# run_if: cell_allowed()
policy, V, converged = value_iteration(env, gamma=1.0, verbose=2)

---------- value_iteration ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0    #  0.0  1.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 1: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  1.0  1.0  0.0 
   1  0.0    #  1.0  1.0 
   2  0.0  0.0  0.0  1.0 
-------------------------
iter 2: delta=1.000000
  r/c   0    1    2    3 
   0  1.0  1.0  1.0  0.0 
   1  0.0    #  1.0  1.0 
   2  0.0  0.0  1.0  1.0 
-------------------------
iter 3: delta=1.000000
  r/c   0    1    2    3 
   0  1.0  1.0  1.0  0.0 
   1  1.0    #  1.0  1.0 
   2  0.0  1.0  1.0  1.0 
-------------------------
iter 4: delta=1.000000
  r/c   0    1    2    3 
   0  1.0  1.0  1.0  0.0 
   1  1.0    #  1.0  1.0 
   2  1.0  1.0  1.0  1.0 
-------------------------
iter 5: delta=0.000000
  r/c   0    1    2    3 
   0  1.0

In [1044]:
# run_if: cell_allowed()
render_policy(env, policy)

 r/c  0   1   2   3 
  0   ↑   ↑   ↑  +1 
  1   ↑   #   ↑   ↑ 
  2   ↑   ↑   ↑   ↑ 


In [1045]:
# run_if: cell_allowed()
# this flipping behaviour is solved by pass_incumbent_policy, in next experiment
policy, V, converged = policy_iteration(env, gamma=1.0, verbose=2)

---------- policy_iteration ------------
policy init
 r/c  0   1   2   3 
  0   ↑   ↑   ↑  +1 
  1   ↑   #   ↑   ↑ 
  2   ↑   ↑   ↑   ↑ 
-------------------------
iter 0: V
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  1.0 
   2  0.0  0.0  0.0  1.0 
-----
actions changed = 3
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   →   ↑ 
  2   ↑   ↑   →   ↑ 
-------------------------
iter 1: V
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0    #  1.0  1.0 
   2  0.0  0.0  1.0  1.0 
-----
actions changed = 5
 r/c  0   1   2   3 
  0   ↑   →   ↑  +1 
  1   ↑   #   ↑   ↑ 
  2   ↑   →   ↑   ↑ 
-------------------------
iter 2: V
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  1.0 
   2  0.0  0.0  0.0  1.0 
-----
actions changed = 5
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   →   ↑ 
  2   ↑   ↑   →   ↑ 
-------------------------
iter 3: V
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0    #  1.0  1.0 

### don't update action until better is available

currently tied action at cell (2, 0), get's flipped from RIGHT to UP, we can make it so that action isn't updated until strictly better is available

In [1046]:
h3 = "don't update action until better is available"

In [1047]:
env = GridWorld(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
)
env.render()

 r/c  0   1   2   3 
  0   ·   ·   ·  +1 
  1   ·   #   ·  -1 
  2   ·   ·   ·   · 


In [1048]:
def read_policy(
    env: GridWorld,
    V: np.ndarray,
    gamma=0.9,
    theta=1e-6,
    incumbent_policy: np.ndarray = None,
):
    policy = np.full(env.nS, -1)
    for s in range(env.nS):
        if env.s2c[s] in env.terminals:
            continue
        q = q_from_v(env, V, s, gamma)
        new_a = int(np.argmax(q))
        if incumbent_policy is not None:
            incumbent_a = incumbent_policy[s]
            if incumbent_a != -1 and q[new_a] - q[incumbent_a] < 10 * theta:
                new_a = incumbent_a
        policy[s] = new_a
    return policy

In [1049]:
def policy_iteration(
    env: GridWorld,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    pass_incumbent_policy=False,
    verbose=0,
    pe_max_iters=1000,
    pe_verbose=0,
):
    policy = np.full(env.nS, UP)
    for cell in env.terminals:
        policy[env.c2s[cell]] = -1
    V = np.zeros(env.nS)

    if verbose >= 2:
        print('---------- policy_iteration ------------')
        print('policy init')
        render_policy(env, policy)

    i = 0
    pe_converged = True
    changed = None
    while changed != 0 and i < max_iters:
        V, pe_converged = policy_evaluation(
            env, policy, gamma, theta, max_iters=pe_max_iters, verbose=pe_verbose
        )
        if not pe_converged:
            break
        new_policy = read_policy(
            env,
            V,
            gamma,
            theta,
            incumbent_policy=policy if pass_incumbent_policy else None,
        )
        changed = int(np.count_nonzero(new_policy != policy))

        if verbose >= 2:
            print('-' * 25)
            print(f'iter {i}: V')
            show_V(env, V)
            print('-' * 5)  
        if verbose >= 1:
            prefix = f'iter {i}: ' if verbose == 1 else ''
            print(f'{prefix}actions changed = {changed}')
        if verbose >= 2:
            render_policy(env, new_policy)
        policy = new_policy
        i += 1
    converged = changed == 0
    if verbose >= 1:
        print('-' * 25)  
        if not pe_converged:
            print(
                f"policy_iteration did not converge because policy_evaluation did not converge"
            )
        elif converged:
            print(f'policy_iteration converged in {i - 1} iterations')
        else:
            print(f'policy_iteration did not converge in {max_iters} iterations')
    return policy, V, converged    

In [1050]:
# run_if: cell_allowed()
policy, V, converged = policy_iteration(env, pass_incumbent_policy=True, verbose=2)

---------- policy_iteration ------------
policy init
 r/c  0   1   2   3 
  0   ↑   ↑   ↑  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ↑ 
-------------------------
iter 0: V
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-----
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ← 
-------------------------
iter 1: V
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0    #  0.9  0.0 
   2  0.0  0.0 0.81 0.73 
-----
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 
-------------------------
iter 2: V
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0    #  0.9  0.0 
   2  0.0 0.73 0.81 0.73 
-----
actions changed = 2
 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   →   →   ↑   ← 
-------------------------
iter 3: V
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 

### policy iteration keeps flipping up/right on cell (0, 2) without pass_incumbent_policy

In [1051]:
h3 = 'policy iteration keeps flipping up/right on cell (0, 2) without pass_incumbent_policy'

In [1052]:
# run_if: cell_allowed()
policy, V, converged = policy_iteration(env, gamma=1.0, verbose=2)

---------- policy_iteration ------------
policy init
 r/c  0   1   2   3 
  0   ↑   ↑   ↑  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ↑ 
-------------------------
iter 0: V
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-----
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ← 
-------------------------
iter 1: V
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0    #  1.0  0.0 
   2  0.0  0.0  1.0  1.0 
-----
actions changed = 4
 r/c  0   1   2   3 
  0   ↑   →   ↑  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   → 
-------------------------
iter 2: V
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-----
actions changed = 3
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   → 
-------------------------
iter 3: V
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0    #  1.0  0.0 

In [1053]:
# run_if: cell_allowed()
policy, V, converged = policy_iteration(
    env, gamma=1.0, verbose=2, pass_incumbent_policy=True, pe_verbose=2
)

---------- policy_iteration ------------
policy init
 r/c  0   1   2   3 
  0   ↑   ↑   ↑  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ↑ 
---------- policy_evaluation ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
iter 1: delta=0.000000
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
policy_evaluation converged in 1 iterations
-------------------------
iter 0: V
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-----
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ← 
---------- policy_evaluation ------------
V init
  r/c   0    1    2    3 
   0  0.0

### pass previous V to policy_evaluation

policy_evaluation throws away its previous V. Every call restarts from np.zeros, which is why the pe_verbose=2 trace shows 4–5 sweeps each time. Warm-starting from the previous evaluation is the standard speedup — the new policy's V is close to the old one's, so it usually converges in 1–2 sweeps.

In [1054]:
h3 = "pass previous V to policy_evaluation"

In [1055]:
def policy_evaluation(
    env: GridWorld,
    policy: np.ndarray,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    verbose=0,
    V: np.ndarray = None
):
    V = np.zeros(env.nS) if V is None else V.copy()
    if verbose >= 2:
        print('---------- policy_evaluation ------------')
        print('V init')
        show_V(env, V)
        print('-' * 25)    
    delta = float('inf')
    i = 0
    while delta >= theta and i < max_iters:
        delta = 0.0
        V_old = V.copy()
        for s in range(env.nS):
            if env.s2c[s] in env.terminals:
                continue
            a = policy[s]
            ns, r = env.step(s, a)
            V[s] = r + gamma * V_old[ns]
            delta = max(delta, abs(V[s] - V_old[s]))
        if verbose >= 1:
            print(f"iter {i}: delta={delta:.6f}")
        if verbose >= 2:
            show_V(env, V)
            print('-' * 25)
        i += 1
    converged = delta < theta
    if verbose >= 1:
        if converged:
            print(f'policy_evaluation converged in {i - 1} iterations')
        else:
            print(f"policy_evaluation did not converge in {max_iters} iterations")
    return V, converged

In [1056]:
def policy_iteration(
    env: GridWorld,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    pass_incumbent_policy=False,
    verbose=0,
    pe_max_iters=1000,
    pe_pass_prev_V=False,
    pe_verbose=0,
):
    policy = np.full(env.nS, UP)
    for cell in env.terminals:
        policy[env.c2s[cell]] = -1
    V = np.zeros(env.nS)

    if verbose >= 2:
        print('---------- policy_iteration ------------')
        print('policy init')
        render_policy(env, policy)

    i = 0
    pe_converged = True
    changed = None
    while changed != 0 and i < max_iters:
        V, pe_converged = policy_evaluation(
            env,
            policy,
            gamma,
            theta,
            max_iters=pe_max_iters,
            verbose=pe_verbose,
            V=V if pe_pass_prev_V else None,
        )
        if not pe_converged:
            break
        new_policy = read_policy(
            env,
            V,
            gamma,
            theta,
            incumbent_policy=policy if pass_incumbent_policy else None,
        )
        changed = int(np.count_nonzero(new_policy != policy))
        if verbose >= 2:
            print('-' * 25)
            print(f'iter {i}: V')
            show_V(env, V)
            print('-' * 5)  
        if verbose >= 1:
            prefix = f'iter {i}: ' if verbose == 1 else ''
            print(f'{prefix}actions changed = {changed}')
        if verbose >= 2:
            render_policy(env, new_policy)
        policy = new_policy
        i += 1
    converged = changed == 0
    if verbose >= 1:
        print('-' * 25)  
        if not pe_converged:
            print(
                f"policy_iteration did not converge because policy_evaluation did not converge"
            )
        elif converged:
            print(f'policy_iteration converged in {i - 1} iterations')
        else:
            print(f'policy_iteration did not converge in {max_iters} iterations')
    return policy, V, converged    

In [1057]:
# run_if: cell_allowed()
policy, V, converged = policy_iteration(
    env, verbose=2, pe_verbose=2, pe_pass_prev_V=True
)

---------- policy_iteration ------------
policy init
 r/c  0   1   2   3 
  0   ↑   ↑   ↑  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ↑ 
---------- policy_evaluation ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
iter 1: delta=0.000000
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
policy_evaluation converged in 1 iterations
-------------------------
iter 0: V
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-----
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ← 
---------- policy_evaluation ------------
V init
  r/c   0    1    2    3 
   0  0.0

In [1058]:
# run_if: cell_allowed()
policy, V, converged = policy_iteration(
    env, verbose=2, pe_verbose=2, pe_pass_prev_V=True, pass_incumbent_policy=True
)

---------- policy_iteration ------------
policy init
 r/c  0   1   2   3 
  0   ↑   ↑   ↑  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ↑ 
---------- policy_evaluation ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
iter 1: delta=0.000000
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
policy_evaluation converged in 1 iterations
-------------------------
iter 0: V
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-----
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ← 
---------- policy_evaluation ------------
V init
  r/c   0    1    2    3 
   0  0.0